# Hospital Readmission Analysis

## Project Overview
Analyze patient and visit data to study readmission patterns across diagnosis, length of stay, demographics, and care factors.

## Learning Objectives
- Calculate readmission rates across clinical and demographic dimensions
- Identify high-risk patient profiles for readmission
- Analyze the relationship between length of stay, discharge planning, and readmission
- Quantify readmission cost and operational impact

## Problem Statement
Hospital leadership needs to reduce 30-day readmissions — a key quality metric tied to reimbursement penalties. Understanding which patients and conditions drive readmissions enables targeted interventions.

## Why This Project Matters
Unplanned readmissions cost the US healthcare system ~$26 billion annually. CMS penalizes hospitals with excess readmission rates, making this both a quality and financial priority.

## Dataset Overview
Synthetic hospital discharge dataset: ~6,000 patient visits with diagnosis, length of stay, demographics, discharge disposition, and readmission status.

## Dataset Source and License Notes
- Synthetic data inspired by MIMIC-style hospital records
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 6000
age = np.clip(np.random.normal(62, 16, n).astype(int), 18, 95)
gender = np.random.choice(['Male', 'Female'], n, p=[0.48, 0.52])
diagnosis = np.random.choice(['Heart Failure', 'Pneumonia', 'COPD', 'Hip Replacement', 'AMI',
                                'Diabetes Complication', 'Sepsis', 'Stroke'],
                               n, p=[0.18, 0.15, 0.12, 0.10, 0.12, 0.13, 0.10, 0.10])
los = np.clip(np.random.lognormal(1.2, 0.6, n), 1, 30).round(0).astype(int)
insurance = np.random.choice(['Medicare', 'Medicaid', 'Private', 'Self-Pay'], n, p=[0.40, 0.18, 0.32, 0.10])
discharge_to = np.random.choice(['Home', 'SNF', 'Home Health', 'Rehab', 'AMA'], n, p=[0.45, 0.20, 0.18, 0.12, 0.05])
comorbidity_count = np.clip(np.random.poisson(2.5, n), 0, 10)
prior_admits_12m = np.clip(np.random.poisson(0.8, n), 0, 8)
ed_visit_prior = (np.random.random(n) < 0.25).astype(int)

# Readmission logic
readmit_prob = 0.05 +     0.08 * (diagnosis == 'Heart Failure') + 0.06 * (diagnosis == 'COPD') +     0.04 * (prior_admits_12m >= 2) + 0.03 * (comorbidity_count >= 4) +     0.05 * (discharge_to == 'AMA') + 0.03 * (los <= 2) +     0.02 * (age >= 75) + 0.03 * ed_visit_prior +     np.random.normal(0, 0.02, n)
readmit_prob = np.clip(readmit_prob, 0.02, 0.50)
readmitted_30d = (np.random.random(n) < readmit_prob).astype(int)

df = pd.DataFrame({
    'PatientID': [f'P{i:05d}' for i in range(n)],
    'Age': age, 'Gender': gender, 'Diagnosis': diagnosis,
    'LOS': los, 'Insurance': insurance, 'DischargeTo': discharge_to,
    'ComorbidityCount': comorbidity_count, 'PriorAdmits12M': prior_admits_12m,
    'EDVisitPrior': ed_visit_prior, 'Readmitted30D': readmitted_30d,
})
df['AgeBand'] = pd.cut(df['Age'], bins=[0, 45, 65, 75, 100], labels=['18-45', '45-65', '65-75', '75+'])

print(f'Dataset: {df.shape}')
print(f'Overall 30-day readmission rate: {df["Readmitted30D"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nDiagnosis distribution:\n{df["Diagnosis"].value_counts()}')
print(f'\nReadmission counts: {df["Readmitted30D"].value_counts().to_dict()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Diagnosis')['Readmitted30D'].mean().sort_values().plot.barh(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Readmission Rate by Diagnosis')

df.groupby('AgeBand')['Readmitted30D'].mean().plot.bar(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Readmission Rate by Age Band')
axes[0,1].tick_params(axis='x', rotation=0)

df.groupby('DischargeTo')['Readmitted30D'].mean().sort_values().plot.barh(ax=axes[1,0], edgecolor='black', color='teal')
axes[1,0].set_title('Readmission Rate by Discharge Disposition')

los_bins = pd.cut(df['LOS'], bins=[0, 2, 4, 7, 14, 99], labels=['1-2', '3-4', '5-7', '8-14', '15+'])
df.groupby(los_bins)['Readmitted30D'].mean().plot.bar(ax=axes[1,1], edgecolor='black', color='goldenrod')
axes[1,1].set_title('Readmission Rate by Length of Stay')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Comorbidity and Prior Admission Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comorb_rate = df.groupby('ComorbidityCount')['Readmitted30D'].mean()
comorb_rate.plot(ax=axes[0], marker='o', color='coral')
axes[0].set_title('Readmission Rate by Comorbidity Count')
axes[0].set_xlabel('Comorbidities')

prior_rate = df.groupby('PriorAdmits12M')['Readmitted30D'].mean()
prior_rate.plot(ax=axes[1], marker='o', color='steelblue')
axes[1].set_title('Readmission Rate by Prior Admissions (12M)')
axes[1].set_xlabel('Prior Admissions')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'comorbidity_prior.png'), dpi=100, bbox_inches='tight')
plt.show()

## Insurance and Readmission

In [ ]:
ins_stats = df.groupby('Insurance').agg(
    patients=('PatientID', 'count'),
    readmit_rate=('Readmitted30D', 'mean'),
    avg_los=('LOS', 'mean'),
    avg_comorbidities=('ComorbidityCount', 'mean'),
).round(3)
print('Readmission by Insurance:')
print(ins_stats.sort_values('readmit_rate', ascending=False))

## High-Risk Profile Summary

In [ ]:
high_risk = df[df['Readmitted30D'] == 1]
low_risk = df[df['Readmitted30D'] == 0]
profile_cols = ['Age', 'LOS', 'ComorbidityCount', 'PriorAdmits12M', 'EDVisitPrior']
comp = pd.DataFrame({
    'Readmitted': high_risk[profile_cols].mean(),
    'Not Readmitted': low_risk[profile_cols].mean(),
}).round(2)
print('Profile Comparison:')
print(comp)

## Interpretation and Business Insight
- **Heart Failure and COPD** have the highest readmission rates — these are known CMS penalty conditions
- **AMA discharges** (Against Medical Advice) have dramatically elevated readmission risk
- **Very short stays** (1-2 days) show higher readmission — possibly premature discharge
- **Comorbidity burden** and **prior admissions** are strong, additive risk factors
- **ED visits** in the prior period signal instability and predict readmission
- **Medicare patients** have higher readmission rates — consistent with older, sicker population

## Limitations
- Synthetic data with simplified readmission logic
- No medication, lab, or vitals data
- No social determinants of health
- No post-discharge follow-up compliance data
- 30-day window is arbitrary — some readmissions are planned

## How to Improve This Project
- Add medication reconciliation and discharge instruction compliance
- Include social determinants (housing, transportation, food security)
- Add post-discharge follow-up call and visit tracking
- Build predictive readmission risk models (LACE, HOSPITAL scores)
- Track readmission by payer penalty vs non-penalty conditions

## Production Considerations
- Real-time readmission risk scoring at discharge
- Care transition nurse assignment for high-risk patients
- Post-discharge follow-up scheduling automation
- Monthly readmission dashboards for CMO and quality teams

## Common Mistakes
- Treating all readmissions as preventable (some are planned or unavoidable)
- Not risk-adjusting when comparing hospitals or units
- Focusing only on inpatient factors while ignoring post-discharge support
- Using overall readmission rate without stratifying by condition

## Mini Challenge / Exercises
1. Calculate the readmission rate for Heart Failure patients with 3+ comorbidities vs those with <3.
2. What is the estimated Medicare penalty if readmission exceeds the national average by 2%?
3. Which discharge disposition has the lowest readmission rate? Why might that be?
4. Build a simple risk score using comorbidity count + prior admits + ED visit flag.

## Final Summary / Key Takeaways
- Hospital readmission analysis identifies modifiable risk factors for targeted intervention
- Diagnosis, comorbidity burden, and prior utilization are the strongest predictors
- Discharge planning (where patients go) significantly affects readmission risk
- Short stays may indicate premature discharge — quality vs efficiency tradeoff
- Proactive care transitions for high-risk patients reduce readmissions and penalties